# ESRS E1-2 — Climate-Related Physical Risk Assessment with CLIMADA

**Identification of Climate-Related Risks and Scenario Analysis**  
*Draft Simplified ESRS (EFRAG Technical Advice V1.5, 30 Nov 2025)*

This notebook runs the full CLIMADA **probabilistic** risk pipeline starting from nothing
but **site geolocations (WGS84 lat/lon)**. Everything else — the hazard event sets — comes
from **open-access data** served by the CLIMADA Data API (ETH Zurich):

| Hazard | Open-access source | CLIMADA `haz_type` |
|---|---|---|
| Flooding (river) | GloFAS / ISIMIP river-flood depth maps | `RF` |
| Extreme precipitation | Derived from the river-flood model (precip → flood → damage) | `RF` |
| Storms (tropical cyclone) | Synthetic tracks from IBTrACS, scaled by CMIP6 | `TC` |
| Wildfires | FIRMS-based wildfire event sets | `WF` |
| Landslides | NASA COOLR / global landslide susceptibility | `LS` |
| Heat Stress / Heatwaves | ISIMIP heat-stress exceedance datasets | `HS` |
| Drought & Water Stress | ISIMIP relative crop-yield loss (rainfed wheat proxy) | `RC` |
| Sea-Level Rise | ISIMIP coastal-flood / SLR inundation maps | `CF` |

The risk equation is **Risk = Hazard × Exposure × Vulnerability**. The output is the
**Expected Annual Loss (EAL)** per site — the core E1-2 metric — computed over a full
probabilistic event set (return-period flood maps / thousands of synthetic storms).

**Run matrix:** 8 hazards × 3 scenarios (SSP1-1.9, SSP2-4.5, SSP3-7.0) × 3 time horizons
= **72 runs per site**.

> Built and tested on **climada 6.1**. Uses `ImpactCalc(...).impact()` (the `Impact.calc()`
> method used in older guides is removed in climada 6.x).
>
> **Data availability note:** Wildfire, landslide, heat stress, drought, and sea-level rise
> datasets have more limited coverage in the CLIMADA open API than flood/TC. Combinations
> that return no data are recorded as `NaN` and flagged in the disclosure table — the
> framework still satisfies E1-2 by documenting the gap and the alternative data source
> needed to fill it.

## 1 · Imports

In [1]:
import pandas as pd
import numpy as np

from climada.entity import Exposures, ImpactFuncSet, ImpactFunc, ImpfTropCyclone
from climada.hazard import Hazard
from climada.engine import ImpactCalc
from climada.util.api_client import Client

## 2 · Load the exposure (your only input: geolocations)

Each row is one site. The **only data you need to supply are `latitude` / `longitude`**.

Because asset replacement values are company-specific (and often the one thing you don't
have yet), `value` defaults to **1.0** for every site. With unit value, the resulting EAL is
the **expected annual loss as a fraction of asset value** (i.e. the mean annual damage
ratio, 0–1). When you obtain real book/replacement values, drop them into the `value`
column and the same EAL becomes a currency figure — no other change needed.

The enrichment columns (`headcount`, `business_area`, `criticality`) are **not** used in the
CLIMADA math; they enrich the E1-2 *sensitivity* narrative (AR 6(b)) and carry through to the
final disclosure table. `country_iso3` is required to fetch the per-country open-access
hazard tiles. `impf_RF` / `impf_TC` link each site to its vulnerability curve (defined below).

In [2]:
# --- The only mandatory inputs: site geolocations (WGS84 decimal degrees) ---
sites = pd.DataFrame({
    'site_name':     ['Manila WH', 'London HQ', 'Singapore DC', 'Paris Office'],
    'latitude':      [14.5995,     51.5074,     1.3521,         48.8566],
    'longitude':     [120.9842,   -0.1278,      103.8198,       2.3522],
    'country_iso3':  ['PHL',       'GBR',        'SGP',          'FRA'],

    # Asset value: default 1.0 -> EAL is expressed as a fraction of asset value.
    # Replace with book / replacement cost (same currency) to get monetary EAL.
    'value':         [1.0,         1.0,          1.0,            1.0],

    # --- Enrichment (post-processing only; not used in CLIMADA's math) ---
    'headcount':     [120,         450,          85,             200],
    'business_area': ['Logistics', 'Corporate HQ','Data Center',  'Sales'],
    'criticality':   ['High',      'Critical',   'Critical',     'Medium'],

    # --- Vulnerability links: which impact-function id applies per hazard type ---
    'impf_RF':       [1, 1, 1, 1],   # river flood / extreme precip  -> ImpactFunc id 1 (RF)
    'impf_TC':       [1, 1, 1, 1],   # tropical cyclone              -> ImpactFunc id 1 (TC)
    'impf_WF':       [1, 1, 1, 1],   # wildfire                      -> ImpactFunc id 1 (WF)
    'impf_LS':       [1, 1, 1, 1],   # landslide                     -> ImpactFunc id 1 (LS)
    'impf_HS':       [1, 1, 1, 1],   # heat stress / heatwave        -> ImpactFunc id 1 (HS)
    'impf_RC':       [1, 1, 1, 1],   # drought / water stress (RC)   -> ImpactFunc id 1 (RC)
    'impf_CF':       [1, 1, 1, 1],   # sea-level rise / coastal flood-> ImpactFunc id 1 (CF)
})

exp = Exposures(sites)
exp.check()
exp.gdf.head()

,site_name,country_iso3,value,headcount,business_area,criticality,impf_RF,impf_TC,impf_WF,impf_LS,impf_HS,impf_RC,impf_CF,geometry
0,Manila WH,PHL,1.0,120,Logistics,High,1,1,1,1,1,1,1,POINT (120.9842 14.5995)
1,London HQ,GBR,1.0,450,Corporate HQ,Critical,1,1,1,1,1,1,1,POINT (-0.1278 51.5074)
2,Singapore DC,SGP,1.0,85,Data Center,Critical,1,1,1,1,1,1,1,POINT (103.8198 1.3521)
3,Paris Office,FRA,1.0,200,Sales,Medium,1,1,1,1,1,1,1,POINT (2.3522 48.8566)


## 3 · Define the 72-run matrix (scenarios × time horizons × hazards)

**Scenarios.** CLIMADA's open hazard API is indexed by RCP labels. We map each SSP to the
closest available RCP that exists for **both** flood and TC hazards:

| ESRS scenario | ~Warming | River flood | Tropical cyclone | Note |
|---|---|---|---|---|
| SSP1-1.9 | +1.5 °C | `rcp26` | `rcp26` | SSP1-2.6 used as conservative proxy (document the ~0.3 °C) |
| SSP2-4.5 | +2.7 °C | `rcp60` | `rcp60` | `rcp45` is absent from the flood dataset → `rcp60` for both |
| SSP3-7.0 | +3.6 °C | `rcp85` | `rcp85` | High-emission scenario (satisfies E1-2 §16(a)(i)) |

**Time horizons.** River flood uses a `year_range` window; tropical cyclone uses a single
`ref_year` snapshot; hazards without scenario projections use `time_prop=None`.

---

### CLIMADA ETH Data API — availability per hazard

The five new hazards do **not** behave like flood/TC in the CLIMADA API. The table below
explains the exact error each produced in the first run and the fix applied in `hazard_config`:

| Hazard | Error observed | Root cause | Fix in this notebook |
|---|---|---|---|
| Flooding | — (works) | Fully supported | Per-country, `year_range` |
| Extreme Precip. | — (works) | Same dataset as Flooding | Per-country, `year_range` |
| Storms | `NoResult` (rcp85+2080 only) | Dataset not published for that combo | Recorded as NaN |
| **Wildfires** | `NoResult` all combos | API has historical FIRMS catalog only — `climate_scenario` / `year_range` filters don't exist for this type | `time_prop=None`: query per-country without scenario or time filters; same historical baseline reused across all 9 cells |
| **Landslides** | `ValueError` all combos | `'landslide'` is **not a registered `haz_type`** in the CLIMADA ETH API — the schema rejects it before any HTTP request | `api_unavailable=True`: skip API; alternatives below |
| **Heat Stress** | `ValueError` all combos | `'heat_stress'` is **not a registered `haz_type`** in the CLIMADA ETH API | `api_unavailable=True`: skip API; alternatives below |
| **Drought & Water Stress** | `NoResult` all combos | Dataset is **global** (no per-country tiles) — `country_iso3alpha` filter returns nothing | `country_independent=True`: single global download, no country filter |
| **Sea-Level Rise** | `ValueError` all combos | `'coastal_flood'` is **not a registered `haz_type`** in the CLIMADA ETH API | `api_unavailable=True`: skip API; alternatives below |

**Alternative sources for `api_unavailable` hazards** (wire these in when data is available):

| Hazard | Recommended alternative | URL |
|---|---|---|
| Landslides | `climada_petals` `Landslide` class with NASA COOLR catalog | https://gpm.nasa.gov/landslides/index.html |
| Landslides | UNDRR ThinkHazard API | https://thinkhazard.org/ |
| Heat Stress | ISIMIP3b WBGT exceedance-day NetCDF → load as custom `Hazard` | https://data.isimip.org/ |
| Heat Stress | Copernicus CDS heat-stress indicators | https://cds.climate.copernicus.eu/ |
| Sea-Level Rise | IPCC AR6 interactive SLR projection tool | https://sealevel.nasa.gov/ipcc-ar6-sea-level-projection-tool |
| Sea-Level Rise | `climada_petals` `CoastSeg` module | https://github.com/CLIMADA-project/climada_petals |
| Sea-Level Rise | WRI AQUEDUCT Coastal | https://www.wri.org/aqueduct |

In [3]:
# SSP -> RCP (intersection that exists for BOTH hazards in the open API)
scenarios = {
    'SSP1-1.9': 'rcp26',   # Paris-aligned (SSP1-2.6 proxy)
    'SSP2-4.5': 'rcp60',   # rcp45 not in flood data -> rcp60 for both
    'SSP3-7.0': 'rcp85',   # high emissions
}

# Per-hazard configuration.
#
#   time_prop          : API property for the time dimension; None = no time filtering
#   timeframes         : ESRS horizon label -> API value (None when time_prop is None)
#   extra_props        : additional fixed API filters (crop_type, model_name, etc.)
#   country_independent: True = global dataset; single download without country_iso3alpha
#   api_unavailable    : True = haz_type not registered in CLIMADA ETH API (ValueError);
#                        load_hazard will look for a local HDF5 built by §5b instead
#   alt_source         : authoritative alternative when api_unavailable=True and §5b not run
hazard_config = {
    'Flooding': {
        'haz_type': 'river_flood', 'tag': 'RF', 'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Extreme Precip.': {
        'haz_type': 'river_flood', 'tag': 'RF', 'impf_col': 'impf_RF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
    },
    'Storms': {
        'haz_type': 'tropical_cyclone', 'tag': 'TC', 'impf_col': 'impf_TC',
        'time_prop': 'ref_year',
        'timeframes': {'Short (0-3yr)': '2040',
                       'Medium (3-10yr)': '2060',
                       'Long (10+yr)': '2080'},
        'extra_props': {'model_name': 'random_walk'},
    },
    'Wildfires': {
        # The CLIMADA ETH API 'wildfire' catalog is historical FIRMS fire-season data.
        # The API returns haz_type='WFseason'; load_hazard normalises this to 'WF' so
        # ImpactCalc matches impf_wf without any changes to the exposure or impf_sets.
        # time_prop=None: no climate_scenario/year_range filters; same baseline reused
        # for all 9 scenario×timeframe cells (conservative for E1-2 screening).
        'haz_type': 'wildfire', 'tag': 'WF', 'impf_col': 'impf_WF',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
    },
    'Landslides': {
        # 'landslide' is NOT registered in the CLIMADA ETH API schema (ValueError).
        # §5b prep cell downloads Felsberg et al. 2022 global landslide susceptibility
        # (Zenodo 6893230, 458 KB) and saves landslide_hist.hdf5.
        # time_prop=None: static susceptibility map — same file reused for all 9 cells.
        'haz_type': 'landslide', 'tag': 'LS', 'impf_col': 'impf_LS',
        'time_prop': None,
        'timeframes': {'Short (0-3yr)': None, 'Medium (3-10yr)': None, 'Long (10+yr)': None},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'Run §5b prep cell (Zenodo 6893230) | climada_petals Landslide + NASA COOLR',
    },
    'Heat Stress': {
        # 'heat_stress' is NOT registered in the CLIMADA ETH API schema (ValueError).
        # §5b prep cell queries Open-Meteo Climate API (free, no key) and saves one
        # HDF5 per scenario × timeframe with mean annual hot-days count (Tmax > 35 °C).
        'haz_type': 'heat_stress', 'tag': 'HS', 'impf_col': 'impf_HS',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'Run §5b prep cell (Open-Meteo Climate API) | ISIMIP3b WBGT (data.isimip.org)',
    },
    'Drought & Water Stress': {
        # relative_cropyield IS in the CLIMADA API but only as a global dataset;
        # per-country queries return NoResult.  country_independent=True issues one
        # global download — if the API still returns NoResult, load_hazard falls back
        # to the local HDF5 written by §5b (Open-Meteo precipitation deficit proxy).
        'haz_type': 'relative_cropyield', 'tag': 'RC', 'impf_col': 'impf_RC',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {'crop_type': 'whe', 'irrigation_type': 'rf'},
        'country_independent': True,
        'alt_source': 'Run §5b prep cell (Open-Meteo precip deficit) | petals Drought (SPEI)',
    },
    'Sea-Level Rise': {
        # 'coastal_flood' is NOT registered in the CLIMADA ETH API schema (ValueError).
        # §5b prep cell uses climada_petals CoastalFlood.from_aqueduct_tif() to download
        # WRI Aqueduct GeoTIFs (~46 MB each, one-time) and saves one HDF5 per scenario
        # × timeframe.  Aqueduct scenario mapping: rcp26 → historical (conservative,
        # no RCP2.6 published), rcp60 → RCP4.5, rcp85 → RCP8.5.
        'haz_type': 'coastal_flood', 'tag': 'CF', 'impf_col': 'impf_CF',
        'time_prop': 'year_range',
        'timeframes': {'Short (0-3yr)': '2010_2030',
                       'Medium (3-10yr)': '2030_2050',
                       'Long (10+yr)': '2050_2070'},
        'extra_props': {},
        'api_unavailable': True,
        'alt_source': 'Run §5b prep cell (WRI Aqueduct via petals) | IPCC AR6 SLR tool (sealevel.nasa.gov)',
    },
}

COUNTRIES = sites['country_iso3'].unique().tolist()
print('Hazards :', list(hazard_config))
print('Scenarios:', list(scenarios))
print('Countries:', COUNTRIES)
print('Total runs:', len(hazard_config) * len(scenarios) * 3)

print('\n--- CLIMADA API / data status per hazard ---')
for _name, _c in hazard_config.items():
    if _c.get('api_unavailable'):
        status = f'NOT in API → §5b builds local HDF5 | {_c["alt_source"]}'
    elif _c.get('country_independent'):
        status = 'global dataset (country_independent=True); §5b local fallback if NoResult'
    elif _c['time_prop'] is None:
        status = 'historical catalog (no scenario/year filters); tag normalised in load_hazard'
    else:
        status = f'fully supported: per-country + {_c["time_prop"]}'
    print(f'  {_name:<28s} {status}')

Hazards : ['Flooding', 'Extreme Precip.', 'Storms', 'Wildfires', 'Landslides', 'Heat Stress', 'Drought & Water Stress', 'Sea-Level Rise']
Scenarios: ['SSP1-1.9', 'SSP2-4.5', 'SSP3-7.0']
Countries: ['PHL', 'GBR', 'SGP', 'FRA']
Total runs: 72

--- CLIMADA API / data status per hazard ---
  Flooding                     fully supported: per-country + year_range
  Extreme Precip.              fully supported: per-country + year_range
  Storms                       fully supported: per-country + ref_year
  Wildfires                    historical catalog (no scenario/year filters); tag normalised in load_hazard
  Landslides                   NOT in API → §5b builds local HDF5 | Run §5b prep cell (Zenodo 6893230) | climada_petals Landslide + NASA COOLR
  Heat Stress                  NOT in API → §5b builds local HDF5 | Run §5b prep cell (Open-Meteo Climate API) | ISIMIP3b WBGT (data.isimip.org)
  Drought & Water Stress       global dataset (country_independent=True); §5b local fallback if NoRe

## 4 · Build the impact (vulnerability) functions

One generic impact function is defined per hazard type. All use `id=1` so every site maps
to the same curve — replace with site-specific IDs if you have differentiated vulnerability
data (e.g., reinforced concrete vs. timber frame for flood).

| Hazard tag | Intensity metric | Curve basis |
|---|---|---|
| `TC` | Max wind speed (m/s) | Emanuel (2011) — field standard |
| `RF` | Flood depth (m) | JRC-style depth-damage |
| `WF` | Fire weather index (FWI) category 0–5 | Generic fire intensity |
| `LS` | Landslide susceptibility index 0–1 | Generic slope-failure damage |
| `HS` | Annual exceedance days above WBGT threshold | Labour/operational disruption |
| `RC` | Fractional crop-yield loss 0–1 (rainfed wheat) | Proportional supply-chain proxy |
| `CF` | Coastal flood / SLR inundation depth (m) | JRC-style depth-damage (same as RF) |

> The hazard tag must match the impact function's `haz_type` **and** the exposure column
> `impf_{TAG}`. The open river-flood datasets use tag **`RF`** (not `FL`), coastal flood
> uses **`CF`**, drought/cropyield uses **`RC`**.

In [4]:
# Tropical cyclone (Emanuel 2011) — haz_type 'TC', id 1
impf_tc = ImpfTropCyclone.from_emanuel_usa()

# River flood / extreme precip — depth (m) -> damage fraction. haz_type 'RF', id 1
impf_rf = ImpactFunc(
    id=1,
    haz_type='RF',
    name='Flood depth-damage (generic)',
    intensity=np.array([0, 0.5, 1, 1.5, 2, 3, 5, 10]),  # flood depth, metres
    mdd=np.array([0, 0.15, 0.30, 0.45, 0.55, 0.70, 0.85, 1.0]),
    paa=np.ones(8),
    intensity_unit='m',
)

# Wildfire — fire weather index (FWI) category -> damage fraction. haz_type 'WF', id 1
impf_wf = ImpactFunc(
    id=1,
    haz_type='WF',
    name='Wildfire (fire weather index)',
    intensity=np.array([0, 1, 2, 3, 4, 5]),   # FWI category 0 (none) – 5 (extreme)
    mdd=np.array([0.0, 0.05, 0.15, 0.35, 0.65, 1.0]),
    paa=np.ones(6),
    intensity_unit='FWI category',
)

# Landslide — susceptibility index (0–1) -> damage fraction. haz_type 'LS', id 1
impf_ls = ImpactFunc(
    id=1,
    haz_type='LS',
    name='Landslide (susceptibility-damage)',
    intensity=np.array([0, 0.25, 0.50, 0.75, 1.0]),   # 0 = none, 1 = very high susceptibility
    mdd=np.array([0.0, 0.10, 0.30, 0.60, 1.0]),
    paa=np.ones(5),
    intensity_unit='susceptibility index (0–1)',
)

# Heat stress / heatwave — annual exceedance days above WBGT threshold -> disruption fraction
# haz_type 'HS', id 1
impf_hs = ImpactFunc(
    id=1,
    haz_type='HS',
    name='Heat stress (labour/operational disruption)',
    intensity=np.array([0, 5, 10, 20, 30, 50, 100]),   # exceedance days per year
    mdd=np.array([0.0, 0.02, 0.05, 0.12, 0.25, 0.50, 0.80]),
    paa=np.ones(7),
    intensity_unit='days/yr above threshold',
)

# Drought / water stress — fractional crop-yield loss (rainfed wheat proxy) -> impact fraction
# haz_type 'RC' (relative_cropyield), id 1
impf_rc = ImpactFunc(
    id=1,
    haz_type='RC',
    name='Drought / water stress (crop-yield proxy)',
    intensity=np.array([0, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0]),  # fractional yield loss
    mdd=np.array([0.0, 0.08, 0.18, 0.32, 0.50, 0.72, 1.0]),
    paa=np.ones(7),
    intensity_unit='fractional yield loss (0–1)',
)

# Sea-level rise / coastal flood — inundation depth (m) -> damage fraction. haz_type 'CF', id 1
impf_cf = ImpactFunc(
    id=1,
    haz_type='CF',
    name='Coastal flood / SLR (depth-damage)',
    intensity=np.array([0, 0.5, 1, 1.5, 2, 3, 5, 10]),   # inundation depth, metres
    mdd=np.array([0, 0.15, 0.30, 0.45, 0.55, 0.70, 0.85, 1.0]),
    paa=np.ones(8),
    intensity_unit='m',
)

impf_sets = {
    'RF': ImpactFuncSet([impf_rf]),
    'TC': ImpactFuncSet([impf_tc]),
    'WF': ImpactFuncSet([impf_wf]),
    'LS': ImpactFuncSet([impf_ls]),
    'HS': ImpactFuncSet([impf_hs]),
    'RC': ImpactFuncSet([impf_rc]),
    'CF': ImpactFuncSet([impf_cf]),
}
print('Impact functions ready:', {k: v.get_ids() for k, v in impf_sets.items()})

Impact functions ready: {'RF': {'RF': [1]}, 'TC': {'TC': [1]}, 'WF': {'WF': [1]}, 'LS': {'LS': [1]}, 'HS': {'HS': [1]}, 'RC': {'RC': [1]}, 'CF': {'CF': [1]}}


## 5 · Hazard loader (open-access, cached)

`load_hazard` operates in one of three modes, selected automatically from `hazard_config`:

| Mode | Flag | When used | Behaviour |
|---|---|---|---|
| **Per-country** | *(default)* | `river_flood`, `tropical_cyclone`, `wildfire` | One API call per country; results merged with `Hazard.concat`. API tag normalised to `cfg['tag']` after fetch (e.g. `WFseason` → `WF`). |
| **Global dataset** | `country_independent=True` | `relative_cropyield` (no country tiles) | One API call without `country_iso3alpha`. Falls back to local §5b HDF5 if `NoResult`. |
| **API unavailable** | `api_unavailable=True` | `landslide`, `heat_stress`, `coastal_flood` | `haz_type` not in CLIMADA ETH schema. Looks for local HDF5 built by §5b; returns `None` if file absent. |

Two additional robustness features apply to all modes:

- **Graceful gaps.** Missing tiles or `NoResult` are skipped and logged; affected sites receive `NaN` rather than crashing the run.
- **In-memory caching.** Each unique `(haz_type, rcp_code, time_value)` triplet is fetched only once. For `time_prop=None` hazards (wildfire, landslide), the cache key collapses all 9 scenario×timeframe cells to a single entry.

> Run §5b (next cell) before §6 to enable landslide, heat stress, drought fallback, and sea-level rise.

In [5]:
import time
from pathlib import Path
from climada.util.api_client import Download
from climada.util.constants import SYSTEM_DIR as _CLIMADA_SYSTEM_DIR

client = Client()
_hazard_cache = {}
_LOCAL_HAZ_BASE = Path(_CLIMADA_SYSTEM_DIR) / 'hazard'

def load_hazard(cfg, rcp_code, time_value):
    """Return a merged Hazard for all COUNTRIES, or None if no data exists.

    Mode 1 — per-country   : one API call per ISO3 country; results merged with Hazard.concat
    Mode 2 — global dataset : one API call without country_iso3alpha (country_independent=True);
                              falls back to local HDF5 if API returns NoResult.
    Mode 3 — API unavailable: haz_type not in CLIMADA ETH schema; looks for a locally
                              prepared HDF5 (built by §5b) before returning None.

    Post-fetch: if the API returns a haz_type that differs from cfg['tag'] (e.g. 'WFseason'
    for the wildfire catalog), the haz_type is normalised to cfg['tag'] so ImpactCalc finds
    the correct impact function without requiring changes to the exposure or impf_sets.
    """
    has_time = (cfg['time_prop'] is not None) and (time_value is not None)
    cache_key = (cfg['haz_type'],
                 rcp_code   if has_time else None,
                 time_value if has_time else None)

    # ── Mode 3: api_unavailable — try local HDF5 prepared by §5b ────────────────
    if cfg.get('api_unavailable', False):
        if cache_key in _hazard_cache:
            return _hazard_cache[cache_key]
        local_dir = _LOCAL_HAZ_BASE / cfg['haz_type']
        if has_time:
            local_file = local_dir / f"{cfg['haz_type']}_{rcp_code}_{time_value}.hdf5"
        else:
            # time-invariant hazards (e.g. landslide susceptibility) share one file
            local_file = local_dir / f"{cfg['haz_type']}_hist.hdf5"
        if local_file.exists():
            print(f'    -> local HDF5: {local_file.name}')
            haz = Hazard.from_hdf5(str(local_file))
            _hazard_cache[cache_key] = haz
            return haz
        return None   # §5b not yet run; run loop records NaN and prints alt_source

    # ── Shared cache check for Modes 1 & 2 ─────────────────────────────────────
    if cache_key in _hazard_cache:
        return _hazard_cache[cache_key]

    def _build_props(iso3=None):
        props = {**cfg['extra_props']}
        if iso3:
            props['country_iso3alpha'] = iso3
        if has_time:
            props['climate_scenario'] = rcp_code
            props[cfg['time_prop']]   = time_value
        return props

    def _normalize_tag(haz):
        """Align haz_type to cfg['tag'] when the API returns a more specific label."""
        if haz is not None and haz.haz_type != cfg['tag']:
            print(f"    - API haz_type '{haz.haz_type}' normalised → '{cfg['tag']}'")
            haz.haz_type = cfg['tag']
        return haz

    # ── Mode 2: global dataset ──────────────────────────────────────────────────
    if cfg.get('country_independent', False):
        haz = None
        for attempt in range(2):
            try:
                haz = client.get_hazard(cfg['haz_type'], properties=_build_props())
                break
            except Exception as exc:
                exc_name = type(exc).__name__
                if exc_name in ('ChunkedEncodingError', 'Failed') and attempt == 0:
                    print(f'    - {exc_name} for global {cfg["haz_type"]} '
                          f'({rcp_code}, {time_value}): retrying...')
                    time.sleep(5)
                else:
                    print(f"    - no {cfg['haz_type']} global "
                          f"({rcp_code}, {time_value}): {exc_name}")
                    break
        # Fallback: try local HDF5 built by §5b if API returned nothing
        if haz is None and has_time:
            local_file = (_LOCAL_HAZ_BASE / cfg['haz_type'] /
                          f"{cfg['haz_type']}_{rcp_code}_{time_value}.hdf5")
            if local_file.exists():
                print(f'    -> local HDF5 fallback: {local_file.name}')
                haz = Hazard.from_hdf5(str(local_file))
        _hazard_cache[cache_key] = _normalize_tag(haz)
        return _hazard_cache[cache_key]

    # ── Mode 1: per-country (standard) ─────────────────────────────────────────
    parts = []
    for iso3 in COUNTRIES:
        for attempt in range(2):
            try:
                parts.append(client.get_hazard(cfg['haz_type'],
                                               properties=_build_props(iso3)))
                break
            except Exception as exc:
                exc_name = type(exc).__name__
                if exc_name in ('ChunkedEncodingError', 'Failed') and attempt == 0:
                    print(f'    - {exc_name} for {iso3} ({rcp_code}, {time_value}): '
                          f'purging stale cache & retrying...')
                    for _d in list(Download.select()):
                        if _d.enddownload is None and cfg['haz_type'] in _d.path:
                            _d.delete_instance()
                    time.sleep(5)
                else:
                    print(f"    - no {cfg['haz_type']} for {iso3} "
                          f"({rcp_code}, {time_value}): {exc_name}")
                    break

    haz = None if not parts else (parts[0] if len(parts) == 1 else Hazard.concat(parts))
    _hazard_cache[cache_key] = _normalize_tag(haz)
    return _hazard_cache[cache_key]

## 5b · Supplementary hazard data preparation

Run the code cell below **once** before §6. It downloads and builds CLIMADA `Hazard` objects
for the four hazards not served by the CLIMADA ETH Data API. Subsequent runs detect existing
HDF5 files and skip any download — re-running the main §6 loop does not re-download.

All files are stored under **`~/climada/data/hazard/<haz_type>/`** — one sub-folder per
hazard type, one HDF5 per scenario × timeframe combination (where the hazard has a time
dimension).

| Hazard | Source | Access | File pattern |
|---|---|---|---|
| **Landslides** | Felsberg et al. 2022 global LSS ensemble mean | Zenodo 6893230 — 458 KB direct download | `landslide/landslide_hist.hdf5` (time-invariant) |
| **Heat Stress** | Open-Meteo Climate API — annual Tmax > 35 °C days | `climate-api.open-meteo.com` — free, no API key | `heat_stress/heat_stress_{rcp}_{yr}.hdf5` |
| **Drought** | Open-Meteo Climate API — annual precipitation deficit vs. 1980–2009 baseline | Same API + ERA5 archive | `relative_cropyield/relative_cropyield_{rcp}_{yr}.hdf5` |
| **Sea-Level Rise** | WRI Aqueduct coastal flood GeoTIFFs via `climada_petals` `CoastalFlood` | S3 auto-download (~46 MB per TIF, one-time) | `coastal_flood/coastal_flood_{rcp}_{yr}.hdf5` |

> **Wildfire** does not need a separate download. The CLIMADA ETH API provides a historical
> FIRMS fire-season dataset (internal `haz_type = 'WFseason'`). `load_hazard` fetches it and
> normalises the tag to `'WF'` automatically so the existing impact function and exposure
> column require no changes.
>
> **Drought** primary path: `load_hazard` still tries the CLIMADA ETH API
> (`relative_cropyield`) first. The Open-Meteo precipitation-deficit file prepared here is
> used only as a local fallback when the API returns `NoResult`.

In [6]:
# ============================================================================
# §5b · Supplementary hazard data preparation
# ============================================================================
# Run ONCE before §6. Safe to re-run — existing HDF5 files are skipped.
# Requires: climada_petals (CoastalFlood), xarray, requests
# ============================================================================

import gc
import requests
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.spatial import cKDTree as _cKDTree
from climada.hazard import Hazard, Centroids

# ── Site centroids ──────────────────────────────────────────────────────────
_lats       = sites['latitude'].values
_lons       = sites['longitude'].values
_site_names = sites['site_name'].values
N_SITES     = len(_lats)
SITE_CENT   = Centroids.from_lat_lon(_lats, _lons)

# ── Timeframe → (year_start, year_end) ─────────────────────────────────────
TIMEFRAME_YEARS = {
    '2010_2030': (2010, 2030),
    '2030_2050': (2030, 2050),
    '2050_2070': (2050, 2070),
}

# ── Helpers ─────────────────────────────────────────────────────────────────

def _ensure_dir(haz_type: str) -> Path:
    p = _LOCAL_HAZ_BASE / haz_type
    p.mkdir(parents=True, exist_ok=True)
    return p

def _make_hazard(haz_type: str, intensity_row: np.ndarray,
                 rcp: str, yr_key: str, freq: float = 1.0,
                 units: str = '') -> Hazard:
    """Minimal CLIMADA Hazard — 1 event × N_SITES centroids."""
    haz = Hazard(haz_type)
    haz.centroids = SITE_CENT
    haz.event_id   = np.array([1])
    haz.event_name = [f'{haz_type}_{rcp}_{yr_key}']
    yr_start = TIMEFRAME_YEARS[yr_key][0]
    haz.date      = np.array([int(pd.Timestamp(f'{yr_start}-07-01').to_julian_date())])
    haz.frequency = np.array([freq])
    haz.intensity = csr_matrix(intensity_row.reshape(1, N_SITES))
    haz.fraction  = csr_matrix(np.ones((1, N_SITES)))
    if units:
        haz.units = units
    haz.check()
    return haz

def _check_api(name: str, url: str) -> bool:
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        ok = r.status_code < 500
        print(f'  {"✓" if ok else "✗"} {name}: HTTP {r.status_code}')
        return ok
    except Exception as e:
        print(f'  ✗ {name}: {type(e).__name__}')
        return False

print('=== §5b · Supplementary hazard data preparation ===')
print(f'Storage root: {_LOCAL_HAZ_BASE}\n')

# ── API reachability ────────────────────────────────────────────────────────
print('--- API reachability ---')
OPEN_METEO_OK = _check_api(
    'Open-Meteo Climate API',
    'https://climate-api.open-meteo.com/v1/climate?latitude=14.6&longitude=120.9'
    '&start_date=2020-01-01&end_date=2020-01-02&models=MRI_AGCM3_2_S&daily=temperature_2m_max')
AQUEDUCT_OK = _check_api(
    'WRI Aqueduct S3',
    'http://wri-projects.s3.amazonaws.com/AqueductFloodTool/download/v2/'
    'inuncoast_historical_nosub_hist_rp0100_0.tif')
ZENODO_LS_OK = _check_api(
    'Zenodo 6893230 (LSS)',
    'https://zenodo.org/api/records/6893230/files/FelsbergEtAl2022_LSS_ensavg.nc4c/content')
print()

# ============================================================================
# 1 · LANDSLIDES
#     Source : Felsberg et al. 2022 ensemble-mean global landslide susceptibility
#     Zenodo : https://zenodo.org/records/6893230  (file: ensavg.nc4c, 458 KB)
#     Hazard : time-invariant susceptibility index (0–1) at each site
#     File   : ~/climada/data/hazard/landslide/landslide_hist.hdf5
# ============================================================================
print('=== 1. Landslides ===')
LS_DIR     = _ensure_dir('landslide')
LS_NC      = LS_DIR / 'FelsbergEtAl2022_LSS_ensavg.nc4c'
LS_OUT     = LS_DIR / 'landslide_hist.hdf5'
LS_ZENODO  = ('https://zenodo.org/api/records/6893230/files/'
              'FelsbergEtAl2022_LSS_ensavg.nc4c/content')

if LS_OUT.exists():
    print(f'  Cached: {LS_OUT}')
elif not ZENODO_LS_OK:
    print('  Zenodo unreachable — skipping landslide prep')
else:
    # Download raster (458 KB)
    if not LS_NC.exists():
        print('  Downloading Felsberg et al. 2022 LSS ensemble mean (458 KB)...')
        r = requests.get(LS_ZENODO, timeout=60)
        r.raise_for_status()
        LS_NC.write_bytes(r.content)
        print(f'  Saved raster: {LS_NC}')

    # Read and extract site susceptibility
    ds = xr.open_dataset(LS_NC)
    var_name = list(ds.data_vars)[0]
    print(f'  LSS variable: {var_name}  shape: {ds[var_name].shape}')

    # Identify coordinate names (lat/lon may be named differently)
    lat_dim = next((c for c in ds.coords if c.lower() in ('lat', 'latitude', 'y')), None)
    lon_dim = next((c for c in ds.coords if c.lower() in ('lon', 'longitude', 'x')), None)

    ls_intensity = np.zeros(N_SITES)
    for i, (lat, lon, sn) in enumerate(zip(_lats, _lons, _site_names)):
        kw = {}
        if lat_dim:
            kw[lat_dim] = lat
        if lon_dim:
            kw[lon_dim] = lon
        try:
            val = float(ds[var_name].sel(**kw, method='nearest').values)
            ls_intensity[i] = float(np.clip(val, 0.0, 1.0))
        except Exception:
            ls_intensity[i] = 0.0
        print(f'    {sn}: LSS = {ls_intensity[i]:.4f}')
    ds.close()

    haz = _make_hazard('LS', ls_intensity, 'hist', '2010_2030',
                       units='susceptibility index (0–1)')
    # Override event name for the time-invariant file
    haz.event_name = ['landslide_historical']
    haz.write_hdf5(str(LS_OUT))
    print(f'  Saved: {LS_OUT}')

# ============================================================================
# 2 · HEAT STRESS
#     Source : Open-Meteo Climate API (CMIP6 model MRI_AGCM3_2_S)
#     Metric : mean annual days with daily max temperature > 35 °C (WBGT proxy)
#     Files  : ~/climada/data/hazard/heat_stress/heat_stress_{rcp}_{yr}.hdf5
# ============================================================================
print('\n=== 2. Heat Stress ===')
HS_DIR = _ensure_dir('heat_stress')
OM_CLIMATE = 'https://climate-api.open-meteo.com/v1/climate'

# RCP → Open-Meteo SSP scenario string
RCP_TO_OM = {'rcp26': 'ssp126', 'rcp60': 'ssp245', 'rcp85': 'ssp370'}

if not OPEN_METEO_OK:
    print('  Open-Meteo unreachable — skipping heat stress prep')
else:
    for rcp, om_ssp in RCP_TO_OM.items():
        for yr_key, (yr_start, yr_end) in TIMEFRAME_YEARS.items():
            out = HS_DIR / f'heat_stress_{rcp}_{yr_key}.hdf5'
            if out.exists():
                print(f'  Cached: {out.name}')
                continue

            print(f'  Querying {rcp}/{yr_key} ({yr_start}–{yr_end}, {om_ssp})...')
            site_hot_days = np.zeros(N_SITES)
            for i, (lat, lon, sn) in enumerate(zip(_lats, _lons, _site_names)):
                params = {
                    'latitude': lat, 'longitude': lon,
                    'start_date': f'{yr_start}-01-01',
                    'end_date':   f'{yr_end}-12-31',
                    'models': 'MRI_AGCM3_2_S',
                    'daily': 'temperature_2m_max',
                    'temperature_unit': 'celsius',
                }
                try:
                    resp = requests.get(OM_CLIMATE, params=params, timeout=90)
                    resp.raise_for_status()
                    j = resp.json()
                    temps = pd.Series(
                        j['daily']['temperature_2m_max'],
                        index=pd.to_datetime(j['daily']['time']),
                    )
                    annual = temps.resample('YE').apply(lambda x: int((x > 35).sum()))
                    site_hot_days[i] = float(annual.mean())
                    print(f'    {sn}: {site_hot_days[i]:.1f} hot days/yr')
                except Exception as exc:
                    print(f'    {sn}: {type(exc).__name__} — set 0')

            haz = _make_hazard('HS', site_hot_days, rcp, yr_key,
                               units='days/yr above 35 °C')
            haz.write_hdf5(str(out))
            print(f'  Saved: {out.name}')

# ============================================================================
# 3 · DROUGHT & WATER STRESS
#     Source : Open-Meteo (Climate API for projections + ERA5 archive for baseline)
#     Metric : fractional annual precipitation deficit vs. 1980–2009 baseline (0–1)
#              used as a proxy for relative crop-yield loss (rainfed wheat)
#     Files  : ~/climada/data/hazard/relative_cropyield/relative_cropyield_{rcp}_{yr}.hdf5
#     Note   : load_hazard tries the CLIMADA ETH API first; this file is the fallback.
# ============================================================================
print('\n=== 3. Drought & Water Stress ===')
RC_DIR = _ensure_dir('relative_cropyield')
OM_ARCHIVE = 'https://archive-api.open-meteo.com/v1/archive'

if not OPEN_METEO_OK:
    print('  Open-Meteo unreachable — skipping drought prep')
else:
    # Step A: historical baseline precipitation per site (ERA5, 1980–2009)
    baseline_path = RC_DIR / 'precip_baseline_1980_2009.csv'
    if baseline_path.exists():
        baseline_df   = pd.read_csv(baseline_path, index_col=0)
        site_baseline = baseline_df['annual_precip_mm'].to_dict()
        print('  Historical baseline loaded from cache.')
    else:
        print('  Fetching ERA5 baseline precipitation 1980–2009...')
        site_baseline = {}
        for lat, lon, sn in zip(_lats, _lons, _site_names):
            params = {
                'latitude': lat, 'longitude': lon,
                'start_date': '1980-01-01', 'end_date': '2009-12-31',
                'daily': 'precipitation_sum',
            }
            try:
                resp = requests.get(OM_ARCHIVE, params=params, timeout=120)
                resp.raise_for_status()
                j = resp.json()
                precip = pd.Series(
                    j['daily']['precipitation_sum'],
                    index=pd.to_datetime(j['daily']['time']),
                ).fillna(0)
                ann = float(precip.resample('YE').sum().mean())
                site_baseline[sn] = ann
                print(f'    {sn}: {ann:.0f} mm/yr baseline')
            except Exception as exc:
                site_baseline[sn] = 600.0   # conservative global mean fallback
                print(f'    {sn}: {type(exc).__name__} — fallback 600 mm/yr')
        pd.DataFrame({'annual_precip_mm': site_baseline}).to_csv(baseline_path)
        print(f'  Baseline cached: {baseline_path.name}')

    # Step B: projected precipitation per scenario × timeframe
    for rcp, om_ssp in RCP_TO_OM.items():
        for yr_key, (yr_start, yr_end) in TIMEFRAME_YEARS.items():
            out = RC_DIR / f'relative_cropyield_{rcp}_{yr_key}.hdf5'
            if out.exists():
                print(f'  Cached: {out.name}')
                continue

            print(f'  Querying {rcp}/{yr_key} ({yr_start}–{yr_end})...')
            site_rc = np.zeros(N_SITES)
            for i, (lat, lon, sn) in enumerate(zip(_lats, _lons, _site_names)):
                params = {
                    'latitude': lat, 'longitude': lon,
                    'start_date': f'{yr_start}-01-01',
                    'end_date':   f'{yr_end}-12-31',
                    'models': 'MRI_AGCM3_2_S',
                    'daily': 'precipitation_sum',
                }
                try:
                    resp = requests.get(OM_CLIMATE, params=params, timeout=90)
                    resp.raise_for_status()
                    j = resp.json()
                    precip = pd.Series(
                        j['daily']['precipitation_sum'],
                        index=pd.to_datetime(j['daily']['time']),
                    ).fillna(0)
                    proj_ann = float(precip.resample('YE').sum().mean())
                    base     = site_baseline.get(sn, 600.0)
                    deficit  = float(np.clip(1 - proj_ann / base, 0.0, 1.0)) if base > 0 else 0.0
                    site_rc[i] = deficit
                    print(f'    {sn}: {proj_ann:.0f} mm/yr projected → '
                          f'deficit={deficit:.3f}')
                except Exception as exc:
                    print(f'    {sn}: {type(exc).__name__} — set 0')

            haz = _make_hazard('RC', site_rc, rcp, yr_key,
                               units='fractional precip deficit (0–1)')
            haz.write_hdf5(str(out))
            print(f'  Saved: {out.name}')

# ============================================================================
# 4 · SEA-LEVEL RISE / COASTAL FLOOD
#     Source : WRI Aqueduct coastal flood GeoTIFFs via climada_petals CoastalFlood
#              http://wri-projects.s3.amazonaws.com/AqueductFloodTool/download/v2/
#     Metric : 1-in-100yr coastal inundation depth (m), per scenario × target year
#     Files  : ~/climada/data/hazard/coastal_flood/coastal_flood_{rcp}_{yr}.hdf5
#
#     Design note: each Aqueduct TIF contains millions of coastal-grid centroids.
#     Calling Hazard.concat() across multiple country hazards would union those
#     centroid sets → MemoryError (127M centroids, 1.9 GB).  Instead we extract
#     site-level intensity from each country hazard immediately using a cKDTree
#     nearest-centroid lookup, then delete the large raster before moving to the
#     next country.  The final saved Hazard has only N_SITES centroids.
# ============================================================================
print('\n=== 4. Sea-Level Rise / Coastal Flood ===')
CF_DIR = _ensure_dir('coastal_flood')

# RCP → Aqueduct (rcp, target_year) mapping.
# Aqueduct has 'historical'/'45'/'85'; no RCP2.6 → use historical as conservative bound.
CF_MAP = {
    'rcp26': [('historical', 'hist', '2010_2030'),
              ('historical', 'hist', '2030_2050'),
              ('historical', 'hist', '2050_2070')],
    'rcp60': [('45', '2030', '2010_2030'),
              ('45', '2050', '2030_2050'),
              ('45', '2080', '2050_2070')],
    'rcp85': [('85', '2030', '2010_2030'),
              ('85', '2050', '2030_2050'),
              ('85', '2080', '2050_2070')],
}

if not AQUEDUCT_OK:
    print('  WRI Aqueduct S3 unreachable — skipping coastal flood prep')
else:
    try:
        from climada_petals.hazard.coastal_flood import CoastalFlood as _CoastalFlood
        PETALS_CF = True
    except ImportError:
        print('  climada_petals not available in this kernel — '
              'activate climada_env_3.11 and re-run')
        PETALS_CF = False

    if PETALS_CF:
        for rcp_code, combos in CF_MAP.items():
            for (aq_rcp, aq_year, yr_key) in combos:
                out = CF_DIR / f'coastal_flood_{rcp_code}_{yr_key}.hdf5'
                if out.exists():
                    print(f'  Cached: {out.name}')
                    continue

                print(f'  {rcp_code}/{yr_key} → Aqueduct rcp={aq_rcp} year={aq_year}')

                # Accumulate site-level intensity; one country at a time to stay in RAM.
                site_cf_intensity = np.zeros(N_SITES)
                any_data = False

                for iso3 in COUNTRIES:
                    try:
                        haz_c = _CoastalFlood.from_aqueduct_tif(
                            rcp=aq_rcp,
                            target_year=aq_year,
                            return_periods=[100],
                            subsidence='wtsub',
                            percentile='95',
                            countries=[iso3],
                        )
                        n_cent = haz_c.centroids.size
                        if n_cent == 0:
                            print(f'    {iso3}: no coastal centroids — skipped')
                            del haz_c
                            continue

                        print(f'    {iso3}: {n_cent} coastal centroids — extracting site values')

                        # Nearest-centroid lookup; skip sites > 1° (~111 km) from coast
                        tree = _cKDTree(haz_c.centroids.coord)   # shape (n_cent, 2) lat/lon
                        for i, (lat, lon, sn) in enumerate(
                                zip(_lats, _lons, _site_names)):
                            dist, idx = tree.query([lat, lon])
                            if dist > 1.0:
                                continue   # inland site; no coastal exposure from this tile
                            col = haz_c.intensity[:, idx].toarray().flatten()
                            val = float(col.max()) if col.size > 0 else 0.0
                            if val > site_cf_intensity[i]:
                                site_cf_intensity[i] = val
                                print(f'      {sn}: {val:.4f} m (from {iso3})')

                        any_data = True

                    except Exception as exc:
                        print(f'    {iso3}: skipped ({type(exc).__name__})')

                    finally:
                        # Always free the large raster before the next country
                        try:
                            del haz_c
                        except NameError:
                            pass
                        gc.collect()

                if any_data:
                    haz = _make_hazard('CF', site_cf_intensity, rcp_code, yr_key,
                                       freq=0.01, units='m')
                    haz.write_hdf5(str(out))
                    print(f'  Saved: {out.name}')
                else:
                    print(f'  No coastal data for any country — {out.name} not written')

# ============================================================================
# Summary
# ============================================================================
print('\n=== Summary — HDF5 files in ~/climada/data/hazard/ ===')
for sub in ['landslide', 'heat_stress', 'relative_cropyield', 'coastal_flood']:
    d = _LOCAL_HAZ_BASE / sub
    files = sorted(d.glob('*.hdf5')) if d.exists() else []
    print(f'  {sub:<22s}: {len(files)} file(s)')
    for f in files[:3]:
        print(f'    {f.name}')
    if len(files) > 3:
        print(f'    … and {len(files)-3} more')

=== §5b · Supplementary hazard data preparation ===
Storage root: C:\Users\engrj\climada\data\hazard

--- API reachability ---
  ✗ Open-Meteo Climate API: ConnectionError
  ✗ WRI Aqueduct S3: ConnectionError
  ✗ Zenodo 6893230 (LSS): ConnectionError

=== 1. Landslides ===
  Cached: C:\Users\engrj\climada\data\hazard\landslide\landslide_hist.hdf5

=== 2. Heat Stress ===
  Open-Meteo unreachable — skipping heat stress prep

=== 3. Drought & Water Stress ===
  Open-Meteo unreachable — skipping drought prep

=== 4. Sea-Level Rise / Coastal Flood ===
  WRI Aqueduct S3 unreachable — skipping coastal flood prep

=== Summary — HDF5 files in ~/climada/data/hazard/ ===
  landslide             : 1 file(s)
    landslide_hist.hdf5
  heat_stress           : 9 file(s)
    heat_stress_rcp26_2010_2030.hdf5
    heat_stress_rcp26_2030_2050.hdf5
    heat_stress_rcp26_2050_2070.hdf5
    … and 6 more
  relative_cropyield    : 9 file(s)
    relative_cropyield_rcp26_2010_2030.hdf5
    relative_cropyield_rcp26

C:\Users\engrj\AppData\Local\Temp\ipykernel_17736\367212062.py:23: DeprecatedWarning: from_lat_lon is deprecated. This method will be removed in a future version. Simply use the constructor instead.
  SITE_CENT   = Centroids.from_lat_lon(_lats, _lons)


## 6 · Run the 72-combination engine

For each (hazard × scenario × horizon) we call `load_hazard` and compute the impact with
`ImpactCalc`. `imp.eai_exp[i]` is the Expected Annual Loss for site *i* (aligned to the
exposure order). Failed or unavailable combinations are recorded with `eal = NaN` so the
disclosure table keeps every row and gaps are explicit rather than silently dropped.

**Prerequisite:** run §5b at least once before this cell to prepare local HDF5 files for
landslide, heat stress, drought (fallback), and sea-level rise. If §5b has not been run,
those hazards will show `-> §5b prep cell not yet run` and record `NaN`.

In [7]:
# Purge stale TC cache entries before the run.
# Queries the CLIMADA Download DB directly — catches entries even when no file
# was written to disk (glob-based purge misses these).
from pathlib import Path
from climada.util.api_client import Download

_purged = 0
_tc_haz_type = hazard_config['Storms']['haz_type']
for _d in list(Download.select()):
    if _d.enddownload is None and _tc_haz_type in _d.path:
        _d.delete_instance()
        print(f'Purged: {Path(_d.path).name}')
        _purged += 1

_hazard_cache.clear()
print(f'\nTotal purged: {_purged}. In-memory cache cleared.')



Total purged: 0. In-memory cache cleared.


In [8]:
from pathlib import Path
from datetime import datetime

all_results = []
n_total = len(hazard_config) * len(scenarios) * 3
n = 0
_api_unavailable_announced = set()   # print alt_source only once per unavailable hazard

for haz_name, cfg in hazard_config.items():
    for ssp_name, rcp_code in scenarios.items():
        for tf_name, tf_value in cfg['timeframes'].items():
            n += 1
            print(f"[{n:2d}/{n_total}] {haz_name} | {ssp_name} | {tf_name}")
            eal_by_site = {}
            try:
                hazard = load_hazard(cfg, rcp_code, tf_value)
                if hazard is None:
                    if cfg.get('api_unavailable', False):
                        if haz_name not in _api_unavailable_announced:
                            print(f'    -> §5b prep cell not yet run — '
                                  f'alt: {cfg.get("alt_source", "")}')
                            _api_unavailable_announced.add(haz_name)
                    elif cfg.get('country_independent', False):
                        if haz_name not in _api_unavailable_announced:
                            print(f'    -> no global API data + no §5b local file; '
                                  f'alt: {cfg.get("alt_source", "")}')
                            _api_unavailable_announced.add(haz_name)
                    else:
                        print('    -> no data for any site; recording NaN')
                else:
                    imp = ImpactCalc(exp, impf_sets[cfg['tag']], hazard).impact(
                        save_mat=False, assign_centroids=True)
                    for i, name in enumerate(sites['site_name']):
                        eal_by_site[name] = float(imp.eai_exp[i])
            except Exception as exc:
                print(f'    -> run failed: {type(exc).__name__}: {exc}')

            for _, row in sites.iterrows():
                all_results.append({
                    'site_name':     row['site_name'],
                    'latitude':      row['latitude'],
                    'longitude':     row['longitude'],
                    'country_iso3':  row['country_iso3'],
                    'headcount':     row['headcount'],
                    'business_area': row['business_area'],
                    'criticality':   row['criticality'],
                    'hazard':        haz_name,
                    'scenario':      ssp_name,
                    'timeframe':     tf_name,
                    'eal':           eal_by_site.get(row['site_name'], np.nan),
                })

base_path = Path('simulations')
today = datetime.now().strftime('%Y%m%d')
prefix = f'{today}_SIM_NUM_'
base_path.mkdir(parents=True, exist_ok=True)

existing = [p for p in base_path.iterdir() if p.is_dir() and p.name.startswith(prefix)]
sim_nums = [int(p.name[len(prefix):]) for p in existing if p.name[len(prefix):].isdigit()]
run_dir = base_path / f'{prefix}{max(sim_nums, default=0) + 1}'
run_dir.mkdir()

results_df = pd.DataFrame(all_results)
out_path = run_dir / 'e1_2_climada_results.csv'
results_df.to_csv(out_path, index=False)
print(f'\nDone. {len(results_df)} rows -> {out_path}')
results_df.head(12)

[ 1/72] Flooding | SSP1-1.9 | Short (0-3yr)
2026-07-01 05:21:31,112 - climada.util.api_client - WARNING - there is no internet connection but the client has stored the results of this very request sometime in the past.
2026-07-01 05:21:31,293 - climada.util.api_client - WARNING - there is no internet connection but the client has stored the results of this very request sometime in the past.
2026-07-01 05:21:31,313 - climada.util.api_client - WARNING - there is no internet connection but the client has stored the results of this very request sometime in the past.
2026-07-01 05:21:31,325 - climada.util.api_client - WARNING - there is no internet connection but the client has stored the results of this very request sometime in the past.
[ 2/72] Flooding | SSP1-1.9 | Medium (3-10yr)
2026-07-01 05:21:31,831 - climada.util.api_client - WARNING - there is no internet connection but the client has stored the results of this very request sometime in the past.
2026-07-01 05:21:31,849 - climada.u

,site_name,latitude,longitude,country_iso3,headcount,business_area,criticality,hazard,scenario,timeframe,eal
0,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Short (0-3yr),0.000051
1,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Short (0-3yr),0.000000
2,Singapore DC,1.3521,103.8198,SGP,85,Data Center,Critical,Flooding,SSP1-1.9,Short (0-3yr),0.000000
3,Paris Office,48.8566,2.3522,FRA,200,Sales,Medium,Flooding,SSP1-1.9,Short (0-3yr),0.001250
4,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Medium (3-10yr),0.000515
5,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
6,Singapore DC,1.3521,103.8198,SGP,85,Data Center,Critical,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
7,Paris Office,48.8566,2.3522,FRA,200,Sales,Medium,Flooding,SSP1-1.9,Medium (3-10yr),0.000000
8,Manila WH,14.5995,120.9842,PHL,120,Logistics,High,Flooding,SSP1-1.9,Long (10+yr),0.000115
9,London HQ,51.5074,-0.1278,GBR,450,Corporate HQ,Critical,Flooding,SSP1-1.9,Long (10+yr),0.000000


## 6 · Run the 72-combination engine

For each (hazard × scenario × horizon) we fetch the open hazard set and compute the impact
with `ImpactCalc`. `imp.eai_exp[i]` is the Expected Annual Loss for site *i* (aligned to the
exposure order). Failed combinations are recorded with `eal = NaN` so the disclosure table
keeps every site/hazard row and the gaps are explicit rather than silently dropped.

For the five new hazards (wildfire, landslide, heat stress, drought, sea-level rise) the
CLIMADA open API has more limited global coverage than flood/TC. `NaN` entries in those rows
signal that an alternative data source (e.g. local government hazard maps, INFORM Risk Index,
or a specialist model) is required to complete the quantitative assessment — this is
documented in §8.

In [9]:
if results_df['eal'].notna().any():
    pivot = results_df.pivot_table(
        index=['site_name', 'business_area', 'criticality', 'headcount', 'hazard'],
        columns=['scenario', 'timeframe'],
        values='eal',
        aggfunc='sum',
        dropna=False,
    )

    site_summary = (results_df.groupby('site_name')
        .agg(total_eal=('eal', 'sum'),
             peak_hazard_eal=('eal', 'max'),
             headcount=('headcount', 'first'),
             criticality=('criticality', 'first'))
        .sort_values('total_eal', ascending=False))

    scenario_summary = (results_df.groupby(['scenario', 'timeframe'])
        .agg(total_eal=('eal', 'sum'),
             max_site_eal=('eal', 'max'),
             sites_at_risk=('eal', lambda s: int((s > 0).sum())))
        .reset_index())

    out_xlsx = run_dir / 'e1_2_disclosure_table.xlsx'
    with pd.ExcelWriter(out_xlsx) as xl:
        pivot.to_excel(xl, sheet_name='EAL_by_site_hazard')
        site_summary.to_excel(xl, sheet_name='Site_summary')
        scenario_summary.to_excel(xl, sheet_name='Scenario_summary')

    print(f'Wrote {out_xlsx} (3 sheets)\n')
    print('=== EAL (fraction of asset value) by site & hazard ===')
    display(pivot.round(4))
    print('\n=== Site summary (most exposed first) ===')
    display(site_summary.round(4))
    print('\n=== Scenario summary ===')
    print(scenario_summary.round(4).to_string(index=False))
else:
    print('No EAL values were produced — check network access to the CLIMADA Data API and '
          'that the requested scenario/year combinations exist for your countries.')

Wrote simulations\20260701_SIM_NUM_1\e1_2_disclosure_table.xlsx (3 sheets)

=== EAL (fraction of asset value) by site & hazard ===


scenario                                                                    SSP1-1.9  \
timeframe                                                               Long (10+yr)   
site_name    business_area criticality headcount hazard                                
London HQ    Corporate HQ  Critical    85        Drought & Water Stress          NaN   
                                                 Extreme Precip.                 NaN   
                                                 Flooding                        NaN   
                                                 Heat Stress                     NaN   
                                                 Landslides                      NaN   
...                                                                              ...   
Singapore DC Sales         Medium      450       Heat Stress                     NaN   
                                                 Landslides                      NaN   
                                                 Sea-Level Rise                  NaN   
                                                 Storms                          NaN   
                                                 Wildfires                       NaN   

scenario                                                                                 \
timeframe                                                               Medium (3-10yr)   
site_name    business_area criticality headcount hazard                                   
London HQ    Corporate HQ  Critical    85        Drought & Water Stress             NaN   
                                                 Extreme Precip.                    NaN   
                                                 Flooding                           NaN   
                                                 Heat Stress                        NaN   
                                                 Landslides                         NaN   
...                                                                                 ...   
Singapore DC Sales         Medium      450       Heat Stress                        NaN   
                                                 Landslides                         NaN   
                                                 Sea-Level Rise                     NaN   
                                                 Storms                             NaN   
                                                 Wildfires                          NaN   

scenario                                                                               \
timeframe                                                               Short (0-3yr)   
site_name    business_area criticality headcount hazard                                 
London HQ    Corporate HQ  Critical    85        Drought & Water Stress           NaN   
                                                 Extreme Precip.                  NaN   
                                                 Flooding                         NaN   
                                                 Heat Stress                      NaN   
                                                 Landslides                       NaN   
...                                                                               ...   
Singapore DC Sales         Medium      450       Heat Stress                      NaN   
                                                 Landslides                       NaN   
                                                 Sea-Level Rise                   NaN   
                                                 Storms                           NaN   
                                                 Wildfires                        NaN   

scenario                                                                    SSP2-4.5  \
timeframe                                                               Long (10+yr)   
site_name    business_area criticality headcount hazard              


=== Site summary (most exposed first) ===


,total_eal,peak_hazard_eal,headcount,criticality
site_name,,,,
Manila WH,1.0607,0.2228,120,High
Paris Office,0.1966,0.0298,200,Medium
Singapore DC,0.1459,0.0597,85,Critical
London HQ,0.0380,0.0126,450,Critical



=== Scenario summary ===
scenario       timeframe  total_eal  max_site_eal  sites_at_risk
SSP1-1.9    Long (10+yr)     0.0120        0.0097              6
SSP1-1.9 Medium (3-10yr)     0.2922        0.2228              9
SSP1-1.9   Short (0-3yr)     0.2495        0.1565             14
SSP2-4.5    Long (10+yr)     0.0143        0.0119              6
SSP2-4.5 Medium (3-10yr)     0.3077        0.2228             11
SSP2-4.5   Short (0-3yr)     0.1859        0.1565             10
SSP3-7.0    Long (10+yr)     0.0066        0.0025              4
SSP3-7.0 Medium (3-10yr)     0.1238        0.0597              9
SSP3-7.0   Short (0-3yr)     0.2492        0.1565             14


## 8 · ESRS E1-2 datapoint mapping

How the outputs above satisfy the draft E1-2 requirements:

| Paragraph | Requirement | Satisfied by |
|---|---|---|
| §14 | Classify each risk as physical / transition | `hazard` column — all eight are **physical** risks |
| §15(a) | Methodology for exposure to hazards | CLIMADA (ETH Zurich, peer-reviewed, used by EIOPA) + this notebook |
| §15(b) | Transition events & trends | **Out of scope** — CLIMADA is physical-risk only; needs a separate assessment |
| §16(a)(i) | ≥1 high-emission scenario | **SSP3-7.0** (`rcp85`) |
| §16(a)(iii) | Temperature projections | SSP→°C table in §3 |
| §16(b) | Scope of operations | `site_name`, `latitude`, `longitude`, `country_iso3` |
| §16(c) | Key assumptions | Impact functions (§4), unit asset value, RCP proxies (§3) |
| §16(d) | Time period of analysis | `timeframe` column / Short–Medium–Long horizons |
| AR 6(a) | Screen which assets are exposed | `eal > 0` flags exposed sites |
| AR 6(b) | Sensitivity considering location | `eal` + `headcount`, `business_area`, `criticality` |

**Documented limitations** (state these in the narrative for audit defensibility):

1. *Extreme precipitation* is derived from the river-flood model (precip → flood → damage),
   per the standard CLIMADA approach — EAL mirrors the flood result.
2. *Tropical cyclone* covers only cyclone-prone basins; European sites (London, Paris) show no
   TC risk. Extratropical windstorm risk for Europe needs the Petals `StormEurope` module.
3. SSP2-4.5 uses `rcp60` (flood data lacks `rcp45`) and SSP1-1.9 uses the SSP1-2.6 proxy;
   both substitutions are conservative and should be disclosed.
4. With `value = 1.0`, EAL is a damage **fraction**; multiply by real asset values for monetary
   loss before reporting figures externally.
5. *Wildfires* — The CLIMADA ETH API `wildfire` type serves a historical FIRMS fire-season
   catalog (internal `haz_type = 'WFseason'`). `load_hazard` normalises the tag to `'WF'`
   automatically. The historical fire-frequency baseline is applied uniformly across all 9
   scenario×timeframe cells (conservative). For scenario-projected wildfire, download
   ISIMIP3b fire data from `data.isimip.org` and load as a custom `Hazard`.
6. *Landslides* — `'landslide'` is not in the CLIMADA ETH API schema. **§5b prep cell**
   downloads the Felsberg et al. 2022 ensemble-mean global landslide susceptibility map
   (Zenodo 6893230, 458 KB) and creates a static `Hazard('LS')` with site-level
   susceptibility index (0–1). The same baseline is applied across all 9 cells (no
   climate-change scaling — susceptibility maps do not have scenario projections yet).
7. *Heat Stress / Heatwaves* — `'heat_stress'` is not in the CLIMADA ETH API schema.
   **§5b prep cell** queries the Open-Meteo Climate API (CMIP6 model MRI_AGCM3_2_S, free,
   no API key) and builds one `Hazard('HS')` per scenario × timeframe with mean annual
   count of days with Tmax > 35 °C as intensity. The 35 °C threshold is a simplified proxy
   for WBGT heat stress; replace with ISIMIP3b WBGT exceedance-day data for higher fidelity.
8. *Drought & Water Stress* — `relative_cropyield` is in the CLIMADA API but only as a
   global dataset; per-country and global queries both return `NoResult` for the scenario
   and year-range combinations available in this notebook. **§5b prep cell** uses the
   Open-Meteo Climate API to compute a fractional annual precipitation deficit
   (projected vs. 1980–2009 ERA5 baseline) as a proxy for crop-yield loss. This is stored
   as `relative_cropyield_{rcp}_{yr}.hdf5` and loaded by `load_hazard` as a fallback.
9. *Sea-Level Rise* — `'coastal_flood'` is not in the CLIMADA ETH API schema. **§5b prep
   cell** uses `climada_petals.CoastalFlood.from_aqueduct_tif()` to download WRI Aqueduct
   coastal flood GeoTIFs (~46 MB per unique scenario × target year, one-time download
   cached by petals). The 1-in-100yr inundation depth (m) is used per country at 95th
   percentile SLR. Aqueduct does not publish RCP2.6 projections; SSP1-1.9 (`rcp26`) uses
   the historical baseline as a conservative lower bound. For higher-precision SLR
   projections, see the IPCC AR6 tool at `sealevel.nasa.gov`.